# Task 18 — Car Price Estimator

This notebook completes the supplied brief: clean the dataset, explore it, train and evaluate a multiple linear regression model, and launch a Gradio prediction interface. The assignment screenshots are treated as requirements; the CSV is the data source.

In [ ]:
%pip -q install pandas scikit-learn matplotlib seaborn gradio

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

# In Colab, upload the CSV if it is not already in the working directory.
CSV_PATH = Path('Car Prices Data.csv')
if not CSV_PATH.exists():
    from google.colab import files
    uploaded = files.upload()
    CSV_PATH = Path(next(iter(uploaded)))


## 1. Inspect and clean

Normalize labels, parse numeric fields, repair the obvious Hyundai typo and malformed two-digit mileage groups, and mark invalid or zero prices as missing. Keep predictor gaps for pipeline imputation; rows without a usable target cannot train a supervised model.

In [ ]:
raw = pd.read_csv(CSV_PATH)
display(raw.head())
print('Shape:', raw.shape)
print('Missing values before cleaning:\n', raw.isna().sum())

def parse_number(value):
    if pd.isna(value): return np.nan
    raw_value = str(value).strip()
    # Repair e.g. '21,14 km' to 21,140 km (a malformed thousands group).
    grouped = re.fullmatch(r'\s*(\d{1,3}),(\d{2})\s*(?:km)?\s*', raw_value, re.I)
    if grouped: return float(grouped.group(1) + grouped.group(2) + '0')
    cleaned = re.sub(r'[^0-9.\-]', '', raw_value.replace(',', ''))
    try: return float(cleaned) if cleaned else np.nan
    except ValueError: return np.nan

df = raw.copy()
df.columns = df.columns.str.strip()
df['Car_Age'] = pd.to_numeric(df['Car_Age'], errors='coerce')
df['Mileage'] = df['Mileage'].map(parse_number)
df['Selling_Price'] = df['Selling_Price'].map(parse_number)
df['Brand'] = df['Brand'].astype(object).str.strip().str.title().replace({'Hyundia': 'Hyundai'})
df['Fuel_Type'] = df['Fuel_Type'].astype(object).str.strip().str.title()
df.loc[df['Selling_Price'] <= 0, 'Selling_Price'] = np.nan
df.loc[df['Mileage'] < 0, 'Mileage'] = np.nan
display(df.head())
print('Missing values after parsing:\n', df.isna().sum())
print('Brands:', sorted(df['Brand'].dropna().unique()))
print('Fuel types:', sorted(df['Fuel_Type'].dropna().unique()))

## 2. Exploratory data analysis

Review distributions and simple relationships. These plots describe this small synthetic/teaching dataset and should not be read as general market trends.

In [ ]:
display(df.describe(include='all').T)
sns.set_theme(style='whitegrid', palette=['#0F8B8D', '#6C63C7', '#F4A261'])
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(data=df, x='Selling_Price', kde=True, ax=axes[0]); axes[0].set_title('Selling price')
sns.scatterplot(data=df, x='Car_Age', y='Selling_Price', ax=axes[1]); axes[1].set_title('Price by car age')
sns.scatterplot(data=df, x='Mileage', y='Selling_Price', ax=axes[2]); axes[2].set_title('Price by mileage')
plt.tight_layout(); plt.show()
plt.figure(figsize=(9, 4))
sns.boxplot(data=df, x='Brand', y='Selling_Price'); plt.xticks(rotation=25); plt.title('Selling price by brand'); plt.tight_layout(); plt.show()
display(df.groupby('Fuel_Type', dropna=False)['Selling_Price'].agg(['count', 'mean', 'median']).round(0))

## 3. Multiple linear regression

Use an 80/20 holdout split. Numeric missing values are median-imputed, categories are mode-imputed and one-hot encoded, and preprocessing is fitted only on the training split. The 1.5×IQR target fence excludes extreme price outliers before splitting. MAE and RMSE are in the dataset's price units; R² can be negative on a small holdout set.

In [ ]:
features = ['Car_Age', 'Mileage', 'Brand', 'Fuel_Type']
target = 'Selling_Price'
model_rows = df.dropna(subset=[target]).copy()
q1, q3 = model_rows[target].quantile([.25, .75]); iqr = q3 - q1
before = len(model_rows)
model_rows = model_rows[model_rows[target].between(q1 - 1.5*iqr, q3 + 1.5*iqr)]
print(f'Target outliers excluded: {before - len(model_rows)}; usable rows: {len(model_rows)}')
X_train, X_test, y_train, y_test = train_test_split(model_rows[features], model_rows[target], test_size=.2, random_state=42)
numeric = ['Car_Age', 'Mileage']; categorical = ['Brand', 'Fuel_Type']
preprocessor = ColumnTransformer([
    ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric),
    ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical),
])
model = Pipeline([('preprocessor', preprocessor), ('regression', LinearRegression())])
model.fit(X_train, y_train)
predictions = model.predict(X_test)
metrics = {'MAE': mean_absolute_error(y_test, predictions), 'RMSE': mean_squared_error(y_test, predictions)**0.5, 'R²': r2_score(y_test, predictions)}
print(pd.Series(metrics).round(3))
plt.figure(figsize=(5, 5)); sns.scatterplot(x=y_test, y=predictions); plt.xlabel('Actual price'); plt.ylabel('Predicted price'); plt.title('Holdout: actual vs predicted')
limits = [min(y_test.min(), predictions.min()), max(y_test.max(), predictions.max())]; plt.plot(limits, limits, '--', color='gray'); plt.tight_layout(); plt.show()

## 4. Gradio estimator

Launch the interface and enter a car's age, mileage, brand, and fuel type. In Colab, `share=True` creates a temporary public link; share it only if appropriate.

In [ ]:
import gradio as gr
brands = sorted(df['Brand'].dropna().unique().tolist())
fuels = sorted(df['Fuel_Type'].dropna().unique().tolist())
def estimate(age, mileage, brand, fuel):
    row = pd.DataFrame([{'Car_Age': age, 'Mileage': mileage, 'Brand': brand, 'Fuel_Type': fuel}], columns=features)
    return f'Estimated selling price: {model.predict(row)[0]:,.0f}'
app = gr.Interface(fn=estimate, inputs=[
    gr.Slider(0, 20, value=5, step=1, label='Car age (years)'),
    gr.Number(value=15000, label='Mileage (km)'),
    gr.Dropdown(brands, label='Brand'), gr.Dropdown(fuels, label='Fuel type')],
    outputs=gr.Textbox(label='Prediction'), title='🚗 Car Price Estimator',
    description=f'Holdout MAE: {metrics["MAE"]:,.0f} | RMSE: {metrics["RMSE"]:,.0f} | R²: {metrics["R²"]:.3f}. Small dataset: educational estimate only.')
app.launch(share=True) # Change to share=False for local-only use